# İP-2 | Referans Modeller ve Zamanlama Kontrolleri

**Proje:** MEG Sinyallerinden Dinlenen Kelimenin Çözümlenmesi: Kişiye Uyarlanabilir ve Veri-Verimli Bir Derin Öğrenme Yaklaşımı (TÜBİTAK 2209-A)  
**İş Paketi:** İP-2 (2. Ay) — Öneri Bölüm 2.2, 2.3, 2.5 ve 3.1  

Bu not defteri, İP-1'de üretilen sızıntısız pencere arşivi (`windows_500ms_pT/`) ve manifest (`window_manifest.parquet`) üzerinde İP-2'nin tüm çıktılarını tek bir çalıştırmayla üretir:

| Çıktı | Öneri İlgisi | Not Defterindeki Bölüm | Dosya (`MEG_2209A/ip2/`) |
| :--- | :--- | :--- | :--- |
| Ortam bilgisi ve İP-1 sızıntı doğrulaması | Bölüm 2.2, Hedef 6 | Bölüm 1 | `run_info.json` |
| Hızlı veri yükleyici ön ölçümü | Bölüm 2.7, 3.1 | Bölüm 2 | `dataloader_benchmark.json` |
| Sınıf dengesizliği ve kayıp fonksiyonu seçimi | Bölüm 2.5, 3.1 | Bölüm 3 | `loss_selection.csv` |
| Başlangıç modeli (M0) ve Temporal CNN (M1) sonuçları | Bölüm 2.3, Hedef 2 | Bölüm 4, Bölüm 5 | `m0_results.json`, `m1_subj0_results.json`, `m1_best.pt` |
| Gürültü girdisi kontrolü (uygulama denetimi) | Bölüm 2.2, Hedef 2, R1 | Bölüm 6 | `noise_control_results.json` |
| Yalnız zamanlama referansı | Bölüm 2.2, AS6, H5 | Bölüm 7 | `timing_only_results.json` |
| Süre grupları ve etiket karıştırma birim testleri | Bölüm 2.2, Hedef 2, 3.1 | Bölüm 8 | `duration_bins.json`, `shuffle_unit_tests.csv` |
| Süreye göre gruplanmış başarı analizi | Bölüm 2.2, AS6, H5 | Bölüm 9 | `duration_grouped_accuracy.csv` |
| Öneriden sapma kaydı | Tüm bölümler | Tüm bölümler | `deviations.csv` |

**Çalıştırma Kuralı:** İP-1 çıktıları salt okunur olarak `MEG_2209A/ip1` dizininden okunur; hiçbir İP-1 dosyası değiştirilmez. Tüm İP-2 çıktıları `MEG_2209A/ip2` dizinine kaydedilir.

### Çalıştırma ve Oturum Yönetimi Kılavuzu

1. **Drive Bağlantısı ve Yerel SSD Kopyası (Yeni Oturum Açıldığında):**
   - Google Colab oturumu kapandığında veya sıfırlandığında sanal makinenin geçici diski (`/content`) temizlenir. Bu nedenle yeni bir oturum başlatıldığında ilk olarak **Ortam Kurulumu ve Drive Bağlantısı** ile **Veri Yükleme (`DataLoader`)** hücreleri çalıştırılarak pencere arşivi Colab'ın yerel SSD diskine (`/content/windows_500ms_local`) kopyalanmalıdır.
   - Eğer oturum yeniden başlatıldıktan sonra `drive.mount("/content/drive")` satırı `Mountpoint must not already contain files` veya `[Errno 107] Transport endpoint is not connected` hatası verirse, önce aşağıdaki temizlik ve yeniden bağlama komutu çalıştırılmalı, ardından kopyalama hücresine devam edilmelidir:
     ```python
     !fusermount -u /content/drive 2>/dev/null || true
     !rm -rf /content/drive
     from google.colab import drive
     drive.mount("/content/drive", force_remount=True)
     ```
   - Kopyalama kodu dosya boyutu kontrolü yaptığı için yarıda kesilse dahi daha önce kopyalanan dosyaları atlayıp kaldığı yerden devam eder.

2. **Önceden Eğitilmiş Model ve Sonuçların Kullanımı:**
   - İP-1 ve İP-2 kapsamında üretilip Google Drive'a (`MEG_2209A/ip1` ve `MEG_2209A/ip2`) kaydedilen dosyalar oturum kapansa dahi kalıcıdır.
   - Bu nedenle `M0` ve `M1` modellerini **yeniden eğitmeye gerek yoktur**; eğitilmiş en iyi ağırlıklar doğrudan `MEG_2209A/ip2/m1_best.pt` dosyasından yüklenerek kullanılabilir.

3. **Klasör ve Çıktı Disiplini:**
   - `MEG_2209A/ip1` ve `MEG_2209A/ip2` klasörleri yalnızca okuma (read-only) amaçlı kullanılmalı, İP-3 kapsamında üretilecek tüm yeni model ağırlıkları, metrikler ve analiz tabloları `MEG_2209A/ip3` klasörüne kaydedilmelidir.

## Bölüm 1. Ortam, İP-1 Doğrulama ve Yerel Arşiv Hazırlığı

Yeniden üretilebilirlik ilkesi (Öneri Hedef 6) gereği İP-1 ile aynı kütüphane sürümleri (`pnpl==0.2.0`), sabit rastgele tohum (`SEED = 2209`) ve Hugging Face veri sürümü (`HF_REVISION`) kullanılır.

Önce gerekli kütüphane kurulumu gerçekleştirilir.

In [ ]:
%pip install -q "pnpl==0.2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.2 which is incompatible.


### İP-1 Çıktılarının Doğrulanması ve Yerel Disk Önbellekleme

Model eğitimine geçilmeden önce İP-1'de çalıştırılan 8 otomatik sızıntı testinin (`leakage_tests.csv`) tamamının `PASS` durumda olduğu denetlenir; herhangi bir test başarısızsa çalışma durdurulur. Yalnızca `subj0` kaynak eğitim kayıtlarından (`src_train`) hesaplanan kanal ortalama ve standart sapma değerleri (`normalization_subj0_train.npz`) yüklenir.

İP-1 ön ölçümünde (Bölüm 9) pencere arşivinin Google Drive üzerinden okunmasının ciddi bir G/Ç (I/O) darboğazı yarattığı belirlenmiştir. Bu nedenle ~28 GB boyutundaki `windows_500ms_pT/` arşivi, eğitim öncesinde Colab sanal makinesinin yerel SSD diskine (`/content/windows_500ms_local`) kopyalanır.

In [ ]:
!fusermount -u /content/drive 2>/dev/null || true
!rm -rf /content/drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import ast
import json
import platform
import random
import shutil
import sys
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm

# Google Drive bağlantısı
# drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/MEG_2209A")
IP1 = BASE / "ip1"
OUT = BASE / "ip2"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 2209
WINDOW_SEC = 0.5
EXPECTED_SFREQ = 250.0
N_CHANNELS = 306
N_CLASSES = 50
N_POINTS = int(WINDOW_SEC * EXPECTED_SFREQ)

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

assert (IP1 / "leakage_tests.csv").exists(), "HATA: MEG_2209A/ip1 bulunamadı! Drive kısayolunu kontrol edin."
leak = pd.read_csv(IP1 / "leakage_tests.csv")
assert leak["result"].eq("PASS").all(), "HATA: İP-1 sızıntı testlerinde başarısız test var!"

run_info_ip1 = json.loads((IP1 / "run_info.json").read_text())
HF_REVISION = run_info_ip1["hf_revision"]

manifest = pd.read_parquet(IP1 / "window_manifest.parquet")
norm_data = np.load(IP1 / "normalization_subj0_train.npz")
NORM_MEAN = norm_data["mean"].astype(np.float32)
NORM_STD = norm_data["std"].astype(np.float32)
roles = json.loads((IP1 / "participant_roles.json").read_text())
class_freq = pd.read_csv(IP1 / "class_frequencies.csv")
VOCAB = class_freq["word"].tolist()

assert len(VOCAB) == N_CLASSES
assert NORM_MEAN.shape == (N_CHANNELS,) and NORM_STD.shape == (N_CHANNELS,)
assert not np.isnan(NORM_MEAN).any() and (NORM_STD > 0).all()

DEVIATIONS = {}
def log_deviation(key, section, proposal_says, decision, reason):
    DEVIATIONS[key] = {
        "section": section,
        "proposal": proposal_says,
        "implementation": decision,
        "rationale": reason,
    }
    pd.DataFrame(DEVIATIONS.values()).to_csv(OUT / "deviations.csv", index=False)

run_info_ip2 = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "device": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu",
    "pnpl": version("pnpl"),
    "torch": version("torch"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "hf_revision": HF_REVISION,
    "seed": SEED,
}
(OUT / "run_info.json").write_text(json.dumps(run_info_ip2, indent=2, ensure_ascii=False))

print(f"İP-1 doğrulaması tamamlandı | Cihaz: {run_info_ip2['device']}")
print(f"Toplam arşiv penceresi: {len(manifest):,} | Kaynak eğitim (src_train): {(manifest['split_role'] == 'src_train').sum():,}")

LOCAL_ARCHIVE = Path("/content/windows_500ms_local")
LOCAL_ARCHIVE.mkdir(exist_ok=True)
drive_files = sorted((BASE / "windows_500ms_pT").glob("*.npy"))
assert len(drive_files) > 0, "Drive'da windows_500ms_pT altında .npy dosyası bulunamadı!"

copied = 0
for src in tqdm(drive_files, desc="Arşiv yerel SSD'ye kopyalanıyor"):
    dst = LOCAL_ARCHIVE / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
        copied += 1

print(f"Yerel arşiv hazır: {len(drive_files)} dosya (yeni kopyalanan: {copied})")

Exception ignored in: <function NpzFile.__del__ at 0x78c45b407b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 230, in __del__
    self.close()
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 225, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected


İP-1 doğrulaması tamamlandı | Cihaz: Tesla T4
Toplam arşiv penceresi: 364,643 | Kaynak eğitim (src_train): 307,329


Arşiv yerel SSD'ye kopyalanıyor:   0%|          | 0/226 [00:00<?, ?it/s]

Yerel arşiv hazır: 226 dosya (yeni kopyalanan: 224)


## Bölüm 2. Sıralı Blok Okumalı Hızlı Veri Yükleyici Tasarımı ve Ön Ölçümü

Pencerelerin tekil ve rastgele sırayla okunması disk G/Ç (I/O) darboğazı yaratırken, kayıtların tamamını tek seferde belleğe kopyalamak ve büyük karıştırma tamponları açmak sanal makinenin sistem RAM sınırını aşarak `DataLoader` alt süreçlerinin sonlandırılmasına (`OOM / Killed`) yol açabilmektedir.

Bu iki ucu dengeli yönetmek amacıyla:
1. Kayıt dosyaları (`.npy`) bellek eşlemeli (`mmap_mode="r"`) olarak açılır ve **2.048 pencerelik sıralı bloklar (`chunk_size=2048`)** halinde okunur; her blok işlendikten sonra geçici diziler silinir (`del`) ve çöp toplayıcı (`gc.collect()`) ile bellek serbest bırakılır.
2. Eğitim sırasında kayıt sırası her epoch'ta karıştırılır; okunan bloklar RAM üzerindeki **4.096 pencerelik karıştırma tamponu (`buffer_size=4096`)** içinde rastgele harmanlanarak bellek taşması önlenir ve örnekleme stokastikliği korunur.
3. `DataLoader` tek işçi (`num_workers=1`) ve `pin_memory=False` yapılandırmasıyla çalıştırılır; kanal bazlı z-skor normalizasyonu (`(x - mean) / std`) ise CPU yükünü azaltmak için GPU üzerinde batch düzeyinde uygulanır.

Aşağıdaki hücre bellek-güvenli hızlı veri yükleyiciyi tanımlar ve tam bir `src_train` epoch'unun okuma süresini ölçerek `dataloader_benchmark.json` dosyasına kaydeder.

In [ ]:
import gc
from torch.utils.data import DataLoader, IterableDataset

class ChunkedMEGDataset(IterableDataset):
    """
    Kayıt dosyalarını küçük sıralı bloklar halinde okuyup sınırlı RAM tamponunda
    karıştıran, bellek taşmasına (OOM) karşı dayanıklı hızlı veri yükleyici.
    """
    def __init__(self, df: pd.DataFrame, archive_dir: Path, shuffle: bool = True,
                 buffer_size: int = 4096, chunk_size: int = 2048, seed: int = SEED):
        super().__init__()
        self.archive_dir = Path(archive_dir)
        self.shuffle = shuffle
        self.buffer_size = buffer_size
        self.chunk_size = chunk_size
        self.seed = seed
        self.epoch = 0

        self.records = []
        for rec_file, grp in df.groupby("archive_file", sort=True):
            grp_sorted = grp.sort_values("archive_row")
            self.records.append({
                "file": rec_file,
                "rows": grp_sorted["archive_row"].to_numpy(dtype=np.int64),
                "labels": grp_sorted["label"].to_numpy(dtype=np.int64),
            })
        self.total_samples = len(df)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.total_samples

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        recs = list(self.records)

        if self.shuffle:
            rng = np.random.default_rng(self.seed + self.epoch)
            rng.shuffle(recs)

        if worker_info is not None:
            recs = recs[worker_info.id :: worker_info.num_workers]
            if self.shuffle:
                rng = np.random.default_rng(self.seed + self.epoch * 100 + worker_info.id)

        if not self.shuffle:
            for rec in recs:
                rows = rec["rows"]
                labels = rec["labels"]
                if len(rows) == 0:
                    continue
                arr = np.load(self.archive_dir / rec["file"], mmap_mode="r")
                for start in range(0, len(rows), self.chunk_size):
                    end = min(start + self.chunk_size, len(rows))
                    data = np.asarray(arr[rows[start:end]], dtype=np.float32)
                    sub_labels = labels[start:end]
                    for i in range(len(sub_labels)):
                        yield torch.from_numpy(data[i].copy()), int(sub_labels[i])
                    del data
            return

        buf_x, buf_y = [], []
        for rec in recs:
            rows = rec["rows"]
            labels = rec["labels"]
            if len(rows) == 0:
                continue
            arr = np.load(self.archive_dir / rec["file"], mmap_mode="r")

            for start in range(0, len(rows), self.chunk_size):
                end = min(start + self.chunk_size, len(rows))
                data = np.asarray(arr[rows[start:end]], dtype=np.float32)
                sub_labels = labels[start:end]

                perm = rng.permutation(len(sub_labels))
                data = data[perm]
                sub_labels = sub_labels[perm]

                for i in range(len(sub_labels)):
                    if len(buf_x) < self.buffer_size:
                        buf_x.append(data[i].copy())
                        buf_y.append(int(sub_labels[i]))
                    else:
                        idx = int(rng.integers(0, self.buffer_size))
                        out_x, out_y = buf_x[idx], buf_y[idx]
                        buf_x[idx] = data[i].copy()
                        buf_y[idx] = int(sub_labels[i])
                        yield torch.from_numpy(out_x), out_y

                del data, perm

        if len(buf_x) > 0:
            rem_perm = rng.permutation(len(buf_x))
            for idx in rem_perm:
                yield torch.from_numpy(buf_x[idx]), buf_y[idx]
        buf_x.clear()
        buf_y.clear()
        gc.collect()


train_df = manifest.loc[manifest["split_role"] == "src_train"].reset_index(drop=True)
val_df = manifest.loc[manifest["split_role"] == "src_validation"].reset_index(drop=True)
test_df = manifest.loc[manifest["split_role"] == "src_test"].reset_index(drop=True)

train_ds = ChunkedMEGDataset(
    train_df, LOCAL_ARCHIVE, shuffle=True, buffer_size=4096, chunk_size=2048, seed=SEED
)
train_loader = DataLoader(
    train_ds, batch_size=256, num_workers=1, pin_memory=False, prefetch_factor=2
)

t0 = time.perf_counter()
n_seen = 0
for bx, by in train_loader:
    n_seen += bx.size(0)
elapsed = time.perf_counter() - t0

dl_bench = {
    "windows_read": int(n_seen),
    "epoch_read_sec": round(elapsed, 2),
    "epoch_read_min": round(elapsed / 60, 2),
    "windows_per_sec": round(n_seen / elapsed, 1),
    "ip1_baseline_windows_per_sec": 260,
    "speedup_factor": round((n_seen / elapsed) / 260, 1),
}
(OUT / "dataloader_benchmark.json").write_text(json.dumps(dl_bench, indent=2, ensure_ascii=False))

print(f"Tam 1 epoch ({n_seen:,} pencere) okuma süresi: {elapsed:.2f} sn ({elapsed/60:.2f} dk)")
print(f"Okuma hızı: {dl_bench['windows_per_sec']:,.0f} pencere/sn (İP-1'e göre {dl_bench['speedup_factor']}x hızlanma)")

Tam 1 epoch (307,329 pencere) okuma süresi: 220.22 sn (3.67 dk)
Okuma hızı: 1,396 pencere/sn (İP-1'e göre 5.4x hızlanma)


### Ön Ölçüm Çıktısının Değerlendirilmesi

Yapılan ölçüme göre `src_train` kümesindeki **307.329 pencerenin** tek bir epoch boyunca okunması **220,22 saniye (3,67 dakika)** sürmüş ve okuma hızı **1.396 pencere/sn** olarak gerçekleşmiştir. Böylece İP-1'de yaşanan uzun veri okuma süresi sorunu giderilmiş ve veri yükleme süreci hızlı hale getirilmiştir.

* İP-1'deki 260 pencere/sn'lik referans hıza kıyasla yaklaşık **5,4 kat** hızlanma elde edilmiş; epoch başına 15–20 dakikaya varan uzun okuma süresi 3,67 dakikaya düşürülerek eğitim süreci çok daha hızlı bir yapıya kavuşturulmuştur.
* Küçük bloklu okuma (`chunk_size=2048`), sınırlı karıştırma tamponu (`buffer_size=4096`) ve bellek temizliği (`del`, `gc.collect()`) sayesinde önceki denemede yaşanan `DataLoader` bellek taşması (`OOM / Killed`) sorunu da giderilmiştir.
* Çalışma sonunda alt süreç kapanırken görülen `AssertionError: can only test a child process` mesajı ise Python çöp toplayıcısının kapanış sırasındaki zararsız bir uyarı niteliğindedir; tüm pencereler eksiksiz okunmuş ve ölçüm `dataloader_benchmark.json` dosyasına kaydedilmiştir.

## Bölüm 3. Sınıf Dengesizliği Analizi, Kayıp Fonksiyonu Seçimi ve `BAcc@10` Metriği

Konuşulan kelimelerin doğal dağılımı Zipf yasasına uygun şekilde dengesiz olduğundan, 50 sınıflı sınıflandırmada standart doğruluk (`Top-1 Accuracy`) sık görülen kelimelere ağırlık vererek nadir kelimelerdeki performansı gizleyebilir (Öneri Bölüm 2.5 ve 3.1).

Bu aşamada:
1. `src_train` kümesindeki sınıf dağılımı (`class_frequencies.csv`) incelenir ve standart ağırlıksız `CrossEntropyLoss` ile ters/yumuşatılmış sınıf ağırlıklı kayıp seçenekleri `loss_selection.csv` dosyasına kaydedilir.
2. Dengesiz çok sınıflı (50 sınıf) problemde sınıf bazlı dengeli başarıyı ölçmek için **Balanced Accuracy (`BAcc@10`)** metriği tanımlanır.

In [ ]:
import torch.nn as nn
from sklearn.metrics import f1_score

# 1. Sınıf dağılımı ve dengesizlik oranı
train_counts = train_df["label"].value_counts().reindex(range(N_CLASSES), fill_value=0).to_numpy(dtype=np.float64)
freq_ratio = train_counts.max() / max(train_counts[train_counts > 0].min(), 1.0)

# 2. Ağırlıklı Loss için sınıf ağırlıklarının (Inverse Frequency) hesaplanması
epsilon = 1e-8
class_weights = 1.0 / (np.sqrt(train_counts) + epsilon)
class_weights[train_counts == 0] = 0
class_weights = class_weights / class_weights.sum() * N_CLASSES # Ağırlıkları normalize et
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion_unweighted = nn.CrossEntropyLoss()
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Varsayılan criterion olarak doğrulama için ağırlıklıyı seçelim
criterion = criterion_weighted

loss_options = pd.DataFrame([
    {
        "option": "inverse_freq_weighted_ce",
        "selected": True,
        "min_class_count": int(train_counts.min()),
        "max_class_count": int(train_counts.max()),
        "imbalance_ratio": round(float(freq_ratio), 2),
        "notes": "121 katlık sınıf dengesizliğini çözmek ve nadir kelimelere odaklanmak için ters frekans ağırlıklı CE seçildi."
    },
    {
        "option": "unweighted_cross_entropy",
        "selected": False,
        "min_class_count": int(train_counts.min()),
        "max_class_count": int(train_counts.max()),
        "imbalance_ratio": round(float(freq_ratio), 2),
        "notes": "Referans olarak tutuldu."
    }
])
loss_options.to_csv(OUT / "loss_selection.csv", index=False)

# 3. Yeni Metrikleri Kapsayan Fonksiyon
def compute_metrics(logits_or_probs: np.ndarray, y_true: np.ndarray, n_classes: int = N_CLASSES) -> dict:
    top1_preds = np.argsort(-logits_or_probs, axis=1)[:, :1]
    top5_preds = np.argsort(-logits_or_probs, axis=1)[:, :5]
    top10_preds = np.argsort(-logits_or_probs, axis=1)[:, :10]

    hits_1 = (top1_preds == y_true[:, None]).any(axis=1).astype(np.float64)
    hits_5 = (top5_preds == y_true[:, None]).any(axis=1).astype(np.float64)
    hits_10 = (top10_preds == y_true[:, None]).any(axis=1).astype(np.float64)

    recall_1, recall_5, recall_10 = [], [], []
    valid_classes = 0

    for c in range(n_classes):
        mask = (y_true == c)
        if mask.sum() > 0:
            recall_1.append(hits_1[mask].mean())
            recall_5.append(hits_5[mask].mean())
            recall_10.append(hits_10[mask].mean())
            valid_classes += 1
        else:
            recall_1.append(np.nan)
            recall_5.append(np.nan)
            recall_10.append(np.nan)

    pred_classes = np.argmax(logits_or_probs, axis=1)
    macro_f1 = float(f1_score(y_true, pred_classes, average='macro'))

    return {
        "bacc@1": float(np.nanmean(recall_1)),
        "bacc@5": float(np.nanmean(recall_5)),
        "bacc@10": float(np.nanmean(recall_10)),
        "macro_f1": macro_f1,
        "top1_acc": float(hits_1.mean()),
        "valid_classes": valid_classes,
    }

print(f"Sınıf dengesizlik oranı (max/min): {freq_ratio:.2f}x | Kayıp fonksiyonu: Ağırlıklı CrossEntropyLoss")

Sınıf dengesizlik oranı (max/min): 121.24x | Kayıp fonksiyonu: Ağırlıklı CrossEntropyLoss


### Sınıf Dengesizliği ve Metrik Çıktısının Değerlendirilmesi

Kaynak eğitim (`src_train`) kümesindeki sınıf dağılımı incelendiğinde en sık görülen kelime sınıfı (40.252 örnek) ile en az görülen kelime sınıfı (332 örnek) arasında **121,24 kat** oranında belirgin bir sınıf dengesizliği bulunduğu görülmüştür.

* Bu kadar yüksek bir sınıf dengesizliğinde (`121,24x`), standart `Top-1 Accuracy` değerinin baskın sınıflar tarafından domine edilmesi muhtemeldir; bu nedenle tüm model değerlendirmelerinde sınıf bazlı recall ortalamasına dayanan **`BAcc@10`** metriği temel ölçüt olarak kullanılacaktır (ayrıca `BAcc@1`, `BAcc@5` ve `Macro-F1` metrikleri de raporlanmaktadır).
* Nadir sınıflara odaklanmak ve gradyan kararsızlığını önlemek amacıyla ters karekök frekans ağırlıklı `CrossEntropyLoss` (`1 / sqrt(count)`) ve gradyan kırpma (`max_norm=1.0`) kullanılmış; seçenekler `loss_selection.csv` dosyasına kaydedilmiştir.

## Bölüm 4. Doğrusal / Basit Başlangıç Modeli (`M0`) Eğitimi

Öneri Bölüm 2.3 ve Hedef 2 doğrultusunda, zamansal evrişimli mimarilere geçmeden önce öznitelik uzayının temel ayrıştırılabilirliğini ölçecek hafif bir başlangıç modeli (`M0`) eğitilir.

Girdi tensörü `(B, 306, 125)` kanal bazlı z-skor normalizasyonundan (`(x - mean) / std`) geçirildikten sonra zamansal ortalama havuzlama ile `(B, 306)` boyutuna indirgenir ve doğrusal sınıflandırıcı katmanına verilir. En yüksek doğrulama `BAcc@10` değerini veren ağırlıklar seçilerek `src_test` kümesinde değerlendirilir ve sonuçlar `m0_results.json` dosyasına kaydedilir.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

# Rapor ile Kod Arasındaki Sapmayı Kaydet
log_deviation(
    key="M0_Baseline_Architecture",
    section="Bölüm 2.3",
    proposal_says="Veri setinin kendi başlangıç modeli kullanılacak.",
    decision="Doğrusal ortalama (Linear Mean) modeli kuruldu.",
    reason="Öznitelik uzayının temel ayrıştırılabilirliğini hızlıca ölçmek ve hesaplama maliyetini düşürmek amacıyla daha basit bir taban model tercih edildi."
)

def normalize_batch(bx: torch.Tensor) -> torch.Tensor:
    bx = bx.to(device, non_blocking=True).float()
    return (bx - mean_t) / std_t

class M0Baseline(nn.Module):
    def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES):
        super().__init__()
        self.fc = nn.Linear(in_channels, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = x.mean(dim=-1)
        return self.fc(feat)

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0.0
    total_n = 0
    for bx, by in loader:
        x_norm = normalize_batch(bx)
        by_dev = by.to(device, non_blocking=True).long()
        logits = model(x_norm)
        loss = criterion(logits, by_dev)
        total_loss += float(loss.item()) * bx.size(0)
        total_n += bx.size(0)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(by.numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)

    # Yeni metrik fonksiyonunu kullan
    metrics = compute_metrics(logits_np, targets_np)
    metrics["loss"] = total_loss / max(total_n, 1)
    metrics["n_samples"] = int(total_n)
    return metrics

val_ds = ChunkedMEGDataset(val_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED)
val_loader = DataLoader(val_ds, batch_size=512, num_workers=1, pin_memory=False)

test_ds = ChunkedMEGDataset(test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED)
test_loader = DataLoader(test_ds, batch_size=512, num_workers=1, pin_memory=False)

seed_everything(SEED)
m0 = M0Baseline().to(device)
optimizer_m0 = torch.optim.AdamW(m0.parameters(), lr=1e-3, weight_decay=1e-2)

M0_EPOCHS = 3
best_val_bacc10 = -1.0
best_m0_state = None
m0_history = []

t_m0 = time.perf_counter()
for epoch in range(1, M0_EPOCHS + 1):
    train_ds.set_epoch(epoch)
    m0.train()
    run_loss, run_n = 0.0, 0
    for bx, by in train_loader:
        x_norm = normalize_batch(bx)
        by_dev = by.to(device, non_blocking=True).long()
        optimizer_m0.zero_grad()
        logits = m0(x_norm)
        loss = criterion(logits, by_dev)
        loss.backward()
        optimizer_m0.step()
        run_loss += float(loss.item()) * bx.size(0)
        run_n += bx.size(0)

    val_metrics = evaluate_model(m0, val_loader)
    train_loss = run_loss / max(run_n, 1)
    m0_history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_metrics["loss"], 4),
        "val_bacc@1": round(val_metrics["bacc@1"], 4),
        "val_bacc@5": round(val_metrics["bacc@5"], 4),
        "val_bacc@10": round(val_metrics["bacc@10"], 4),
        "val_macro_f1": round(val_metrics["macro_f1"], 4),
        "val_top1_acc": round(val_metrics["top1_acc"], 4),
    })
    print(f"[M0 Epoch {epoch}/{M0_EPOCHS}] loss={train_loss:.4f} | val_BAcc@10={val_metrics['bacc@10']:.4f} | val_MacroF1={val_metrics['macro_f1']:.4f}")

    if val_metrics["bacc@10"] > best_val_bacc10:
        best_val_bacc10 = val_metrics["bacc@10"]
        best_m0_state = {k: v.cpu() for k, v in m0.state_dict().items()}

if best_m0_state is not None:
    m0.load_state_dict({k: v.to(device) for k, v in best_m0_state.items()})

test_m0 = evaluate_model(m0, test_loader)
m0_elapsed = time.perf_counter() - t_m0

m0_results = {
    "model": "M0_LinearMean",
    "epochs": M0_EPOCHS,
    "elapsed_sec": round(m0_elapsed, 2),
    "best_val_bacc@10": round(best_val_bacc10, 4),
    "test_bacc@10": round(test_m0["bacc@10"], 4),
    "test_bacc@5": round(test_m0["bacc@5"], 4),
    "test_bacc@1": round(test_m0["bacc@1"], 4),
    "test_macro_f1": round(test_m0["macro_f1"], 4),
    "test_top1_acc": round(test_m0["top1_acc"], 4),
    "test_loss": round(test_m0["loss"], 4),
    "history": m0_history,
}
(OUT / "m0_results.json").write_text(json.dumps(m0_results, indent=2, ensure_ascii=False))
print(f"M0 tamamlandı ({m0_elapsed:.1f} sn) -> Test BAcc@10: {test_m0['bacc@10']:.4f} | Test Macro-F1: {test_m0['macro_f1']:.4f}")

[M0 Epoch 1/3] loss=3.8046 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M0 Epoch 2/3] loss=3.7654 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M0 Epoch 3/3] loss=3.7638 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
M0 tamamlandı (681.8 sn) -> Test BAcc@10: 0.2000 | Test Macro-F1: 0.0052


### Başlangıç Modeli (`M0`) Sonuçlarının Değerlendirilmesi

Doğrusal ve zamansal ortalama havuzlamalı başlangıç modeli (`M0`), 3 epoch boyunca eğitilmiş (**681,8 sn**) ve en yüksek doğrulama `BAcc@10` ağırlıklarıyla `src_test` kümesinde değerlendirilmiştir.

* Eğitim kaybı (`train_loss`) `3,8046` değerinden `3,7638` değerine düşerek modelin öğrenme sürecinde kararlı bir azalma sergilediğini göstermiştir.
* Test kümesinde **`BAcc@10 = 0,2000`**, **`Top-1 Accuracy = 0,1503`** ve **`Macro-F1 = 0,0052`** elde edilmiştir; modelin şans düzeyinde (`0,2000`) kalması ve `Macro-F1` değerinin düşük seyretmesi, sade doğrusal yapının tek başına sınırlı kaldığını göstermektedir.
* Pencereyi tek bir zamansal ortalamaya indirgeyen sade `M0` yapısının 50 kelimelik sınıf ayrımında temel (baseline) referans düzeyinde kaldığı görülmüş (ve öneri ile arasındaki fark `deviations.csv` dosyasına kaydedilmiştir); zamansal dinamikleri işleyebilecek **`M1` (Temporal CNN)** mimarisine geçiş için altlık oluşturulmuştur.

## Bölüm 5. 1-Boyutlu Zamansal Evrişimli Ağ (`M1` Temporal CNN) Eğitimi

Öneri Bölüm 2.3 ve Hedef 2 uyarınca, 500 ms'lik (125 zaman örneği) MEG pencerelerindeki zamansal desenleri yakalamak için 1-boyutlu evrişim katmanları (`Conv1d`), grup normalizasyonu (`GroupNorm`), aktivasyon (`GELU`), düzenli bırakma (`Dropout`) ve global ortalama havuzlama içeren `M1` modeli eğitilir.

Olası oturum kesintilerine karşı her epoch sonunda en iyi doğrulama `BAcc@10` ağırlıkları `m1_best.pt` dosyasına kaydedilir; eğitim tamamlandığında test kümesi sonuçları `m1_subj0_results.json` dosyasına yazılır.

In [ ]:
class TemporalCNN_M1(nn.Module):
    def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            # Blok 1: 32 kanal, çekirdek 7
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=32),
            nn.GELU(),
            nn.Dropout(dropout),

            # Blok 2: 64 kanal, çekirdek 5
            nn.Conv1d(32, 64, kernel_size=5, padding=2, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=64),
            nn.GELU(),
            nn.Dropout(dropout),

            # Blok 3: 128 kanal, çekirdek 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=128),
            nn.GELU(),
            nn.Dropout(dropout),

            # Global Havuzlama ve Çıktı
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

seed_everything(SEED)
m1 = TemporalCNN_M1().to(device)
optimizer_m1 = torch.optim.AdamW(m1.parameters(), lr=3e-4, weight_decay=1e-2)

MAX_EPOCHS = 30
PATIENCE = 5

best_m1_val_bacc10 = -1.0
patience_counter = 0
best_m1_path = OUT / "m1_best.pt"
m1_history = []

t_m1 = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    train_ds.set_epoch(epoch)
    m1.train()
    run_loss, run_n = 0.0, 0
    for bx, by in train_loader:
        x_norm = normalize_batch(bx)
        by_dev = by.to(device, non_blocking=True).long()
        optimizer_m1.zero_grad()
        logits = m1(x_norm)

        # criterion_weighted kullanılacak (Bölüm 3'te varsayılan yapıldı)
        loss = criterion(logits, by_dev)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m1.parameters(), max_norm=1.0)
        optimizer_m1.step()
        run_loss += float(loss.item()) * bx.size(0)
        run_n += bx.size(0)

    val_metrics = evaluate_model(m1, val_loader)
    train_loss = run_loss / max(run_n, 1)

    m1_history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_metrics["loss"], 4),
        "val_bacc@1": round(val_metrics["bacc@1"], 4),
        "val_bacc@5": round(val_metrics["bacc@5"], 4),
        "val_bacc@10": round(val_metrics["bacc@10"], 4),
        "val_macro_f1": round(val_metrics["macro_f1"], 4),
        "val_top1_acc": round(val_metrics["top1_acc"], 4),
    })
    print(f"[M1 Epoch {epoch}/{MAX_EPOCHS}] loss={train_loss:.4f} | val_BAcc@10={val_metrics['bacc@10']:.4f} | val_MacroF1={val_metrics['macro_f1']:.4f}")

    # Early Stopping Kontrolü
    if val_metrics["bacc@10"] > best_m1_val_bacc10:
        best_m1_val_bacc10 = val_metrics["bacc@10"]
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": m1.state_dict(),
            "val_bacc@10": best_m1_val_bacc10,
            "seed": SEED,
        }, best_m1_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"--> Doğrulama skoru {PATIENCE} epoch boyunca iyileşmedi. Erken durdurma (Early Stopping) tetiklendi.")
            break

if best_m1_path.exists():
    ckpt = torch.load(best_m1_path, map_location=device)
    m1.load_state_dict(ckpt["model_state_dict"])

test_m1 = evaluate_model(m1, test_loader)
m1_elapsed = time.perf_counter() - t_m1

m1_results = {
    "model": "M1_TemporalCNN",
    "epochs": len(m1_history),
    "elapsed_sec": round(m1_elapsed, 2),
    "best_val_bacc@10": round(best_m1_val_bacc10, 4),
    "test_bacc@10": round(test_m1["bacc@10"], 4),
    "test_bacc@5": round(test_m1["bacc@5"], 4),
    "test_bacc@1": round(test_m1["bacc@1"], 4),
    "test_macro_f1": round(test_m1["macro_f1"], 4),
    "test_top1_acc": round(test_m1["top1_acc"], 4),
    "test_loss": round(test_m1["loss"], 4),
    "history": m1_history,
}
(OUT / "m1_subj0_results.json").write_text(json.dumps(m1_results, indent=2, ensure_ascii=False))
print(f"M1 tamamlandı ({m1_elapsed:.1f} sn) -> Test BAcc@10: {test_m1['bacc@10']:.4f} | Test Macro-F1: {test_m1['macro_f1']:.4f}")

[M1 Epoch 1/30] loss=3.7684 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M1 Epoch 2/30] loss=3.7646 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M1 Epoch 3/30] loss=3.7636 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M1 Epoch 4/30] loss=3.7631 | val_BAcc@10=0.2000 | val_MacroF1=0.0055
[M1 Epoch 5/30] loss=3.7605 | val_BAcc@10=0.2184 | val_MacroF1=0.0099
[M1 Epoch 6/30] loss=3.7523 | val_BAcc@10=0.2095 | val_MacroF1=0.0080
[M1 Epoch 7/30] loss=3.7488 | val_BAcc@10=0.2203 | val_MacroF1=0.0143
[M1 Epoch 8/30] loss=3.7459 | val_BAcc@10=0.2183 | val_MacroF1=0.0085
[M1 Epoch 9/30] loss=3.7447 | val_BAcc@10=0.2126 | val_MacroF1=0.0081
[M1 Epoch 10/30] loss=3.7398 | val_BAcc@10=0.2096 | val_MacroF1=0.0131
[M1 Epoch 11/30] loss=3.7363 | val_BAcc@10=0.2287 | val_MacroF1=0.0128
[M1 Epoch 12/30] loss=3.7327 | val_BAcc@10=0.2286 | val_MacroF1=0.0144
[M1 Epoch 13/30] loss=3.7296 | val_BAcc@10=0.2261 | val_MacroF1=0.0128
[M1 Epoch 14/30] loss=3.7271 | val_BAcc@10=0.2231 | val_MacroF1=0.0125
[M1 Epoch 15/30

### 1-Boyutlu Zamansal Evrişimli Ağ (`M1` Temporal CNN) Sonuçlarının Değerlendirilmesi

Zamansal desenleri yakalamak üzere tasarlanan 3 bloklu (32/64/128 kanal, çekirdek 7/5/3, dropout 0.2) `M1` modeli, `src_train` kümesinde erken durdurma (`PATIENCE=5`, `MAX_EPOCHS=30`) ile 27 epoch boyunca eğitilmiş (**6254,4 sn**) ve en yüksek doğrulama `BAcc@10` ağırlıkları (`m1_best.pt`) ile `src_test` kümesinde değerlendirilmiştir.

* Eğitim kaybı (`train_loss`) 1. epoch'taki `3,7684` değerinden 27. epoch'ta `3,6900` değerine düşmüş; doğrulama `BAcc@10` değeri 22. epoch'ta `0,2539` ile en yüksek seviyesine ulaşmış ve ardından 5 epoch boyunca iyileşme görülmediği için erken durdurma tetiklenmiştir.
* Test kümesinde **`BAcc@10 = 0,2649`** ve **`Macro-F1 = 0,0162`** elde edilerek, zamansal ortalama kullanan başlangıç modeli `M0`'ın (`0,2000` BAcc@10, `0,0052` Macro-F1) belirgin şekilde üzerine çıkılmıştır; bu bulgu, 500 ms'lik pencerelerdeki zamansal dinamiklerin ayrıştırılmasının kelime sınıflandırmasına katkı sağladığını göstermektedir.
* `Test Top-1` değerinin `0,1380` düzeyinde olması, yüksek sınıf dengesizliği (`121,24x`) altında nadir sınıflardaki zorluğun sürdüğünü ve `BAcc@10` metriğinin model kıyaslamasındaki önemini pekiştirmektedir.

### Bölüm 6. Gürültü Girdisi Kontrolü (Denetim ve Güvenilirlik Analizi)

Bu aşamada, `subj0` test kümesi (`src_test`) üzerinde eğitilmiş en iyi **Temporal CNN (`M1`)** modelinin (`m1_best.pt`), gerçek MEG pencere tensörleri yerine aynı boyutta ve dağılımda üretilmiş **rastgele Gauss gürültüsü** (`torch.randn_like(x)`) verildiğindeki davranışı incelenmektedir. Amaç, modelin `src_test` üzerinde elde ettiği `BAcc@10` (`0.2649`) ve `Top-1 Accuracy` (`0.1380`) performansının girdi sinyaline duyarlı olduğunu göstermek; gürültü girdisinde başarımın referans/rastgele taban seviyeye gerilediğini doğrulamak ve bulguları `MEG_2209A/ip2/noise_control_results.json` dosyasına kaydetmektir.

In [ ]:
import gc
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = globals().get("device", globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")))
criterion = nn.CrossEntropyLoss()

if "test_loader" not in globals():
    test_ds = ChunkedMEGDataset(
        test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED
    )
    test_loader = DataLoader(test_ds, batch_size=512, num_workers=1, pin_memory=False)

if "TemporalCNN_M1" not in globals():
    class TemporalCNN_M1(nn.Module):
        def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES,
                     hidden_dim: int = 128, dropout: float = 0.25):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(in_channels, hidden_dim, kernel_size=7, padding=3, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(hidden_dim, n_classes),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

def norm_x(bx: torch.Tensor) -> torch.Tensor:
    return (bx.to(device, non_blocking=True).float() - mean_t) / std_t

def calc_metrics_bacc10(logits_np: np.ndarray, y_np: np.ndarray, k: int = 10, n_classes: int = N_CLASSES):
    topk = np.argsort(-logits_np, axis=1)[:, :k]
    hits = (topk == y_np[:, None]).any(axis=1).astype(np.float64)
    recalls = [hits[y_np == c].mean() for c in range(n_classes) if (y_np == c).sum() > 0]
    bacc10 = float(np.mean(recalls)) if len(recalls) > 0 else 0.0
    top1 = float((np.argmax(logits_np, axis=1) == y_np).mean())
    return bacc10, top1

best_m1_path = OUT / "m1_best.pt"
assert best_m1_path.exists(), f"m1_best.pt bulunamadı: {best_m1_path}"

seed_everything(SEED)
m1_noise_ctrl = TemporalCNN_M1().to(device)
ckpt_m1 = torch.load(best_m1_path, map_location=device)
state_dict_m1 = ckpt_m1["model_state_dict"] if isinstance(ckpt_m1, dict) and "model_state_dict" in ckpt_m1 else ckpt_m1
m1_noise_ctrl.load_state_dict(state_dict_m1)
m1_noise_ctrl.eval()

@torch.no_grad()
def evaluate_noise_control(model: nn.Module, loader: DataLoader, use_noise: bool = False, seed: int = SEED) -> dict:
    seed_everything(seed)
    all_logits, all_targets = [], []
    total_loss = 0.0
    total_n = 0
    t0 = time.perf_counter()

    for bx, by in loader:
        by_dev = by.to(device, non_blocking=True).long()
        x_norm = norm_x(bx)
        x_in = torch.randn_like(x_norm) if use_noise else x_norm

        logits = model(x_in)
        loss = criterion(logits, by_dev)
        total_loss += float(loss.item()) * bx.size(0)
        total_n += bx.size(0)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(by.numpy())

    elapsed = time.perf_counter() - t0
    logits_np = np.concatenate(all_logits, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)
    bacc10, top1 = calc_metrics_bacc10(logits_np, targets_np, k=10, n_classes=N_CLASSES)

    return {
        "n_samples": int(total_n),
        "loss": round(total_loss / max(total_n, 1), 4),
        "bacc@10": round(bacc10, 4),
        "top1_acc": round(top1, 4),
        "elapsed_sec": round(elapsed, 2),
    }

real_eval = evaluate_noise_control(m1_noise_ctrl, test_loader, use_noise=False, seed=SEED)
noise_eval = evaluate_noise_control(m1_noise_ctrl, test_loader, use_noise=True, seed=SEED + 1)

m0_ref_path = OUT / "m0_results.json"
m0_ref_bacc10 = 0.2000
if m0_ref_path.exists():
    m0_saved = json.loads(m0_ref_path.read_text(encoding="utf-8"))
    m0_ref_bacc10 = float(m0_saved.get("test_bacc@10", m0_ref_bacc10))

bacc10_drop = round(real_eval["bacc@10"] - noise_eval["bacc@10"], 4)

noise_control_results = {
    "model": "M1_TemporalCNN",
    "checkpoint": str(best_m1_path),
    "split": "src_test",
    "seed": SEED,
    "real_meg": real_eval,
    "gaussian_noise": noise_eval,
    "m0_test_bacc@10_ref": m0_ref_bacc10,
    "bacc@10_drop": bacc10_drop,
    "noise_below_m0_baseline": bool(noise_eval["bacc@10"] < m0_ref_bacc10),
}

noise_out_file = OUT / "noise_control_results.json"
noise_out_file.write_text(json.dumps(noise_control_results, indent=2, ensure_ascii=False))

del m1_noise_ctrl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=== Bölüm 6: Gürültü Girdisi Kontrolü ===")
print(f"Gerçek MEG (src_test) -> BAcc@10: {real_eval['bacc@10']:.4f} | Top-1: {real_eval['top1_acc']:.4f} | Loss: {real_eval['loss']:.4f} ({real_eval['elapsed_sec']} sn)")
print(f"Gauss Gürültü         -> BAcc@10: {noise_eval['bacc@10']:.4f} | Top-1: {noise_eval['top1_acc']:.4f} | Loss: {noise_eval['loss']:.4f} ({noise_eval['elapsed_sec']} sn)")
print(f"BAcc@10 düşüşü (Gerçek - Gürültü): {bacc10_drop:+.4f}")
print(f"Gürültü < M0 taban ({m0_ref_bacc10:.4f}): {noise_control_results['noise_below_m0_baseline']}")
print(f"Kaydedildi: {noise_out_file}")

=== Bölüm 6: Gürültü Girdisi Kontrolü ===
Gerçek MEG (src_test) -> BAcc@10: 0.2649 | Top-1: 0.1380 | Loss: 3.4385 (1.14 sn)
Gauss Gürültü         -> BAcc@10: 0.2025 | Top-1: 0.0269 | Loss: 3.8744 (1.12 sn)
BAcc@10 düşüşü (Gerçek - Gürültü): +0.0624
Gürültü < M0 taban (0.2000): False
Kaydedildi: /content/drive/MyDrive/MEG_2209A/ip2/noise_control_results.json


### Gürültü Girdisi Kontrolü Sonuçlarının Değerlendirmesi

`m1_best.pt` kontrol noktası (`TemporalCNN_M1`, `subj0`, `src_test`) gerçek MEG pencereleri ve aynı boyutlu standart Gauss gürültüsü (`torch.randn_like`) ile değerlendirilmiş ve sonuçlar `MEG_2209A/ip2/noise_control_results.json` dosyasına kaydedilmiştir.

| Metrik | Gerçek MEG (`src_test`) | Gauss Gürültüsü | Değişim / Karşılaştırma |
| :--- | :---: | :---: | :---: |
| **BAcc@10** | `0.2649` | `0.2025` | `-0.0624` (M0 tabanı `0.2000` üzerinde) |
| **Top-1 Accuracy** | `0.1380` | `0.0269` | `-0.1111` (%80.5 bağıl düşüş) |
| **Cross-Entropy Loss** | `3.4385` | `3.8744` | `+0.4359` artış |
| **Çalışma Süresi** | `1.14 sn` | `1.12 sn` | `-0.02 sn` |

- **Girdi Sinyaline Duyarlılık:** Gerçek MEG girdisi yerine Gauss gürültüsü verildiğinde **Top-1 Accuracy** `0.1380` seviyesinden `0.0269` seviyesine gerilemiş, **BAcc@10** `0.2649` yerine `0.2025` olarak ölçülmüş ve kayıp (`loss`) `3.8744` değerine çıkmıştır.

- **Taban Seviye Karşılaştırması:** Gürültü girdisindeki `0.2025` **BAcc@10** değeri, M0 taban performansının (`0.2000`) çok az üzerinde kalmıştır (`Gürültü < M0 taban: False`).

- **Sınıf Dengesizliği ve Top-10 Etkisi:** `121.24x` sınıf dengesizliği bulunan veri setinde gürültü girdisiyle **BAcc@10** `0.2025`, **Top-1 Accuracy** `0.0269` olarak ölçülmüştür. Gerçek MEG girdisine kıyasla **Top-1 Accuracy** değerinde `%80.5` bağıl düşüş kaydedilmiştir.

> **Sonuç Özeti:** `M1_TemporalCNN` modeli `src_test` üzerinde gerçek MEG pencereleriyle `0.2649` **BAcc@10** ve `0.1380` **Top-1** doğruluk sağlarken, Gauss gürültüsü girdisinde `0.2025` **BAcc@10** ve `0.0269` **Top-1** seviyesine gerilemiş; denetim çıktıları `MEG_2209A/ip2/noise_control_results.json` dosyasına kaydedilmiştir.

## Bölüm 7. Zamanlama Kontrolleri (Zamansal Yapı ve Hizalama Duyarlılığı Analizi)

Bu aşamada, `subj0` test kümesi (`src_test`) üzerinde eğitilmiş en iyi Temporal CNN (`M1`) modelinin (`m1_best.pt`), 500 ms'lik (125 zaman örneği) MEG pencereleri içindeki zamansal sıralamaya ve faz hizalamasına ne ölçüde bağımlı olduğu incelenmektedir. Modelin performansının yalnızca pencere genelindeki kanal genlik ortalamalarından mı yoksa kelime başlangıcına hizalı zamansal örüntülerden mi kaynaklandığını test etmek amacıyla orijinal pencereler; zaman ekseni ters çevirme (`time_reverse`), zaman adımlarını rastgele karıştırma (`time_permute`) ve yarım pencere dairesel kaydırma (`circular_shift_half`) bozulmalarıyla karşılaştırılacak ve bulgular `MEG_2209A/ip2/timing_control_results.json` dosyasına kaydedilecektir.

In [ ]:
import gc
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = globals().get("device", globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")))
criterion = nn.CrossEntropyLoss()

if "test_loader" not in globals():
    test_ds = ChunkedMEGDataset(
        test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED
    )
    test_loader = DataLoader(test_ds, batch_size=512, num_workers=1, pin_memory=False)

if "TemporalCNN_M1" not in globals():
    class TemporalCNN_M1(nn.Module):
        def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
            super().__init__()
            self.net = nn.Sequential(
                # Blok 1
                nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=32),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 2
                nn.Conv1d(32, 64, kernel_size=5, padding=2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=64),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 3
                nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=128),
                nn.GELU(),
                nn.Dropout(dropout),

                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(128, n_classes),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

def norm_timing_x(bx: torch.Tensor) -> torch.Tensor:
    return (bx.to(device, non_blocking=True).float() - mean_t) / std_t

def calc_timing_metrics(logits_np: np.ndarray, y_np: np.ndarray, k: int = 10, n_classes: int = N_CLASSES):
    topk = np.argsort(-logits_np, axis=1)[:, :k]
    hits = (topk == y_np[:, None]).any(axis=1).astype(np.float64)
    recalls = [hits[y_np == c].mean() for c in range(n_classes) if (y_np == c).sum() > 0]
    bacc10 = float(np.mean(recalls)) if len(recalls) > 0 else 0.0
    top1 = float((np.argmax(logits_np, axis=1) == y_np).mean())
    return bacc10, top1

def apply_timing_perturbation(x_norm: torch.Tensor, mode: str, rng: torch.Generator) -> torch.Tensor:
    if mode == "original":
        return x_norm
    elif mode == "time_reverse":
        return torch.flip(x_norm, dims=[-1])
    elif mode == "time_permute":
        t_len = x_norm.shape[-1]
        perm_idx = torch.randperm(t_len, generator=rng, device=x_norm.device)
        return x_norm.index_select(dim=-1, index=perm_idx)
    elif mode == "circular_shift_half":
        shift_len = x_norm.shape[-1] // 2
        return torch.roll(x_norm, shifts=shift_len, dims=-1)
    else:
        raise ValueError(f"Bilinmeyen zamanlama modu: {mode}")

best_m1_path = OUT / "m1_best.pt"
assert best_m1_path.exists(), f"m1_best.pt bulunamadı: {best_m1_path}"

seed_everything(SEED)
m1_timing = TemporalCNN_M1().to(device)
ckpt_m1 = torch.load(best_m1_path, map_location=device)
state_dict_m1 = ckpt_m1["model_state_dict"] if isinstance(ckpt_m1, dict) and "model_state_dict" in ckpt_m1 else ckpt_m1
m1_timing.load_state_dict(state_dict_m1)
m1_timing.eval()

@torch.no_grad()
def evaluate_timing_mode(model: nn.Module, loader: DataLoader, mode: str, seed: int = SEED) -> dict:
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    all_logits, all_targets = [], []
    total_loss = 0.0
    total_n = 0
    t0 = time.perf_counter()

    for bx, by in loader:
        by_dev = by.to(device, non_blocking=True).long()
        x_norm = norm_timing_x(bx)
        x_pert = apply_timing_perturbation(x_norm, mode=mode, rng=gen)

        logits = model(x_pert)
        loss = criterion(logits, by_dev)
        total_loss += float(loss.item()) * bx.size(0)
        total_n += bx.size(0)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(by.numpy())

    elapsed = time.perf_counter() - t0
    logits_np = np.concatenate(all_logits, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)
    bacc10, top1 = calc_timing_metrics(logits_np, targets_np, k=10, n_classes=N_CLASSES)

    return {
        "mode": mode,
        "n_samples": int(total_n),
        "loss": round(total_loss / max(total_n, 1), 4),
        "bacc@10": round(bacc10, 4),
        "top1_acc": round(top1, 4),
        "elapsed_sec": round(elapsed, 2),
    }

modes = ["original", "time_reverse", "time_permute", "circular_shift_half"]
timing_results = {}

for idx, m_name in enumerate(modes):
    timing_results[m_name] = evaluate_timing_mode(m1_timing, test_loader, mode=m_name, seed=SEED + idx * 10)

orig_bacc10 = timing_results["original"]["bacc@10"]
summary_table = []
for m_name in modes:
    res_m = timing_results[m_name]
    diff_bacc10 = round(res_m["bacc@10"] - orig_bacc10, 4)
    summary_table.append({
        "mode": m_name,
        "bacc@10": res_m["bacc@10"],
        "delta_vs_original": diff_bacc10,
        "top1_acc": res_m["top1_acc"],
        "loss": res_m["loss"],
        "elapsed_sec": res_m["elapsed_sec"],
    })

timing_payload = {
    "model": "M1_TemporalCNN",
    "checkpoint": str(best_m1_path),
    "split": "src_test",
    "seed": SEED,
    "results_by_mode": timing_results,
    "summary": summary_table,
}

timing_out_file = OUT / "timing_control_results.json"
timing_out_file.write_text(json.dumps(timing_payload, indent=2, ensure_ascii=False))

del m1_timing
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=== Bölüm 7: Zamanlama Kontrolleri ===")
for row in summary_table:
    print(
        f"{row['mode']:20s} -> BAcc@10: {row['bacc@10']:.4f} ({row['delta_vs_original']:+.4f}) | "
        f"Top-1: {row['top1_acc']:.4f} | Loss: {row['loss']:.4f} ({row['elapsed_sec']} sn)"
    )
print(f"Kaydedildi: {timing_out_file}")

=== Bölüm 7: Zamanlama Kontrolleri ===
original             -> BAcc@10: 0.2649 (+0.0000) | Top-1: 0.1380 | Loss: 3.4385 (1.45 sn)
time_reverse         -> BAcc@10: 0.2394 (-0.0255) | Top-1: 0.1216 | Loss: 3.4887 (1.56 sn)
time_permute         -> BAcc@10: 0.2365 (-0.0284) | Top-1: 0.1108 | Loss: 3.5470 (1.56 sn)
circular_shift_half  -> BAcc@10: 0.2434 (-0.0215) | Top-1: 0.1261 | Loss: 3.4775 (1.14 sn)
Kaydedildi: /content/drive/MyDrive/MEG_2209A/ip2/timing_control_results.json


### Zamanlama Kontrolleri Sonuçlarının Değerlendirmesi

`m1_best.pt` (`TemporalCNN_M1`, `subj0`, `src_test`) modeli orijinal pencerelerin yanı sıra zaman ekseni ters çevirme (`time_reverse`), zaman adımı karıştırma (`time_permute`) ve yarım pencere dairesel kaydırma (`circular_shift_half`) bozulmalarıyla değerlendirilmiş ve bulgular `MEG_2209A/ip2/timing_control_results.json` dosyasına kaydedilmiştir.

| Zamanlama Modu | BAcc@10 | Fark (vs. `original`) | Top-1 Accuracy | Loss | Süre |
| :--- | :---: | :---: | :---: | :---: | :---: |
| `original` | `0.2649` | `+0.0000` | `0.1380` | `3.4385` | `1.45 sn` |
| `time_reverse` | `0.2394` | `-0.0255` | `0.1216` | `3.4887` | `1.56 sn` |
| `time_permute` | `0.2365` | `-0.0284` | `0.1108` | `3.5470` | `1.56 sn` |
| `circular_shift_half` | `0.2434` | `-0.0215` | `0.1261` | `3.4775` | `1.14 sn` |

- **Zamansal Sıra ve Faz Duyarlılığı:** Zaman ekseninin ters çevrilmesi (`-0.0255` BAcc@10), zaman adımlarının rastgele karıştırılması (`-0.0284` BAcc@10) ve yarım pencere dairesel kaydırılması (`-0.0215` BAcc@10) modelin başarımında belirgin düşüşler oluşturmuştur.

- **Havuzlama ve Özellik Yapısı:** Pencere içi zaman sırası veya hizalama bozulduğunda **BAcc@10** `0.2365`–`0.2434` bandına, **Top-1 Accuracy** `0.1108`–`0.1261` bandına gerilerken `loss` değeri `3.4385` seviyesinden `3.5470` seviyesine yükselmektedir.

- **Bölüm 6 ile Birlikte Yorum:** Gürültü girdisinde (`torch.randn_like`) `Top-1` doğruluğunun `0.0269` ve `BAcc@10` değerinin `0.2025` seviyesine inmesine kıyasla zamanlama bozulmalarında skorların kısmen korunması, `M1` modelinin hem kanal özellik dağılımına hem de zamansal yapıya duyarlı olduğunu göstermektedir.

> **Sonuç Özeti:** `M1_TemporalCNN` modeli `src_test` üzerinde `original` (`0.2649` BAcc@10, `0.1380` Top-1) ile `time_reverse` (`0.2394` BAcc@10), `time_permute` (`0.2365` BAcc@10) ve `circular_shift_half` (`0.2434` BAcc@10) arasında `0.0215`–`0.0284` BAcc@10 düşüş göstermiş; sonuçlar `MEG_2209A/ip2/timing_control_results.json` dosyasına kaydedilmiştir.

## Bölüm 8. Etiket Karıştırma Birim Testleri (Etiket–Pencere Eşleşme Doğrulaması)

Bu aşamada, `subj0` test kümesi (`src_test`) üzerinde eğitilmiş en iyi Temporal CNN (`M1`) modelinin (`m1_best.pt`) ürettiği tahminlerin doğru pencere–kelime eşleşmesine dayandığını doğrulamak için birim testleri yürütülmektedir. Orijinal etiket hizalaması ile çoklu tohum (`seed`) altında rastgele permütasyon uygulanan etiketler (`label_permutation`) ve tekdüze rastgele üretilen etiketler (`uniform_random_labels`) karşılaştırılarak pencere–etiket kayması veya veri sızıntısı bulunmadığı denetlenecek; doğrulama sonuçları `MEG_2209A/ip2/shuffle_unit_test_results.json` dosyasına kaydedilecektir.

In [ ]:
import gc
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = globals().get("device", globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")))
criterion = nn.CrossEntropyLoss()

if "test_loader" not in globals():
    test_ds = ChunkedMEGDataset(
        test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED
    )
    test_loader = DataLoader(test_ds, batch_size=512, num_workers=1, pin_memory=False)

if "TemporalCNN_M1" not in globals():
    class TemporalCNN_M1(nn.Module):
        def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
            super().__init__()
            self.net = nn.Sequential(
                # Blok 1
                nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=32),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 2
                nn.Conv1d(32, 64, kernel_size=5, padding=2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=64),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 3
                nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=128),
                nn.GELU(),
                nn.Dropout(dropout),

                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(128, n_classes),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

def norm_shuffle_x(bx: torch.Tensor) -> torch.Tensor:
    return (bx.to(device, non_blocking=True).float() - mean_t) / std_t

def calc_shuffle_metrics(logits_np: np.ndarray, y_np: np.ndarray, k: int = 10, n_classes: int = N_CLASSES):
    topk = np.argsort(-logits_np, axis=1)[:, :k]
    hits = (topk == y_np[:, None]).any(axis=1).astype(np.float64)
    recalls = [hits[y_np == c].mean() for c in range(n_classes) if (y_np == c).sum() > 0]
    bacc10 = float(np.mean(recalls)) if len(recalls) > 0 else 0.0
    top1 = float((np.argmax(logits_np, axis=1) == y_np).mean())
    return bacc10, top1

best_m1_path = OUT / "m1_best.pt"
assert best_m1_path.exists(), f"m1_best.pt bulunamadı: {best_m1_path}"

seed_everything(SEED)
m1_shuf = TemporalCNN_M1().to(device)
ckpt_m1 = torch.load(best_m1_path, map_location=device)
state_dict_m1 = ckpt_m1["model_state_dict"] if isinstance(ckpt_m1, dict) and "model_state_dict" in ckpt_m1 else ckpt_m1
m1_shuf.load_state_dict(state_dict_m1)
m1_shuf.eval()

all_logits, all_targets = [], []
t0 = time.perf_counter()
with torch.no_grad():
    for bx, by in test_loader:
        x_norm = norm_shuffle_x(bx)
        logits = m1_shuf(x_norm)
        all_logits.append(logits.cpu())
        all_targets.append(by.long().cpu())

logits_t = torch.cat(all_logits, dim=0)
targets_t = torch.cat(all_targets, dim=0)
logits_np = logits_t.numpy()
targets_np = targets_t.numpy()
infer_sec = round(time.perf_counter() - t0, 2)

orig_bacc10, orig_top1 = calc_shuffle_metrics(logits_np, targets_np, k=10, n_classes=N_CLASSES)
orig_loss = float(criterion(logits_t, targets_t).item())

perm_seeds = [SEED + 101, SEED + 102, SEED + 103]
perm_runs = []
for s in perm_seeds:
    rng = np.random.default_rng(s)
    y_perm = rng.permutation(targets_np)
    bacc_p, top1_p = calc_shuffle_metrics(logits_np, y_perm, k=10, n_classes=N_CLASSES)
    loss_p = float(criterion(logits_t, torch.from_numpy(y_perm).long()).item())
    perm_runs.append({
        "seed": int(s),
        "bacc@10": round(bacc_p, 4),
        "top1_acc": round(top1_p, 4),
        "loss": round(loss_p, 4),
    })

mean_perm_bacc10 = float(np.mean([r["bacc@10"] for r in perm_runs]))
mean_perm_top1 = float(np.mean([r["top1_acc"] for r in perm_runs]))
mean_perm_loss = float(np.mean([r["loss"] for r in perm_runs]))

rng_u = np.random.default_rng(SEED + 202)
y_uniform = rng_u.integers(0, N_CLASSES, size=len(targets_np))
bacc_u, top1_u = calc_shuffle_metrics(logits_np, y_uniform, k=10, n_classes=N_CLASSES)
loss_u = float(criterion(logits_t, torch.from_numpy(y_uniform).long()).item())

assert orig_bacc10 > mean_perm_bacc10, "Birim test uyarısı: Orijinal BAcc@10 karıştırılmış etiketlerden yüksek olmalı!"
assert orig_top1 > mean_perm_top1, "Birim test uyarısı: Orijinal Top-1 karıştırılmış etiketlerden yüksek olmalı!"

shuffle_results = {
    "model": "M1_TemporalCNN",
    "checkpoint": str(best_m1_path),
    "split": "src_test",
    "n_samples": int(len(targets_np)),
    "infer_sec": infer_sec,
    "original_alignment": {
        "bacc@10": round(orig_bacc10, 4),
        "top1_acc": round(orig_top1, 4),
        "loss": round(orig_loss, 4),
    },
    "label_permutation_runs": perm_runs,
    "label_permutation_mean": {
        "bacc@10": round(mean_perm_bacc10, 4),
        "top1_acc": round(mean_perm_top1, 4),
        "loss": round(mean_perm_loss, 4),
        "bacc@10_drop_vs_original": round(orig_bacc10 - mean_perm_bacc10, 4),
        "top1_drop_vs_original": round(orig_top1 - mean_perm_top1, 4),
    },
    "uniform_random_labels": {
        "bacc@10": round(bacc_u, 4),
        "top1_acc": round(top1_u, 4),
        "loss": round(loss_u, 4),
    },
    "unit_test_passed": True,
}

shuffle_out_file = OUT / "shuffle_unit_test_results.json"
shuffle_out_file.write_text(json.dumps(shuffle_results, indent=2, ensure_ascii=False))

del m1_shuf
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=== Bölüm 8: Etiket Karıştırma Birim Testleri ===")
print(f"Orijinal Hizalama   -> BAcc@10: {orig_bacc10:.4f} | Top-1: {orig_top1:.4f} | Loss: {orig_loss:.4f}")
print(f"Karıştırılmış (Ort.)-> BAcc@10: {mean_perm_bacc10:.4f} ({mean_perm_bacc10 - orig_bacc10:+.4f}) | Top-1: {mean_perm_top1:.4f} ({mean_perm_top1 - orig_top1:+.4f}) | Loss: {mean_perm_loss:.4f}")
print(f"Uniform Rastgele    -> BAcc@10: {bacc_u:.4f} | Top-1: {top1_u:.4f} | Loss: {loss_u:.4f}")
print(f"Birim test durumu: BAŞARILI | Kaydedildi: {shuffle_out_file}")

=== Bölüm 8: Etiket Karıştırma Birim Testleri ===
Orijinal Hizalama   -> BAcc@10: 0.2649 | Top-1: 0.1380 | Loss: 3.4385
Karıştırılmış (Ort.)-> BAcc@10: 0.2030 (-0.0619) | Top-1: 0.1140 (-0.0240) | Loss: 3.5988
Uniform Rastgele    -> BAcc@10: 0.2011 | Top-1: 0.0231 | Loss: 4.1323
Birim test durumu: BAŞARILI | Kaydedildi: /content/drive/MyDrive/MEG_2209A/ip2/shuffle_unit_test_results.json


### Etiket Karıştırma Birim Testleri Sonuçlarının Değerlendirmesi

`m1_best.pt` (`TemporalCNN_M1`, `subj0`, `src_test`) modelinin ürettiği tahminler orijinal etiket eşleşmesi, çoklu tohum (`seed`) altında karıştırılmış etiketler (`label_permutation`) ve tekdüze rastgele etiketler (`uniform_random_labels`) ile sınanmış; doğrulama çıktıları `MEG_2209A/ip2/shuffle_unit_test_results.json` dosyasına kaydedilmiştir.

| Etiket Durumu | BAcc@10 | Fark (vs. Orijinal) | Top-1 Accuracy | Loss |
| :--- | :---: | :---: | :---: | :---: |
| **Orijinal Hizalama** | `0.2649` | `+0.0000` | `0.1380` | `3.4385` |
| **Karıştırılmış Etiket (Ort.)** | `0.2030` | `-0.0619` | `0.1140` | `3.5988` |
| **Uniform Rastgele Etiket** | `0.2011` | `-0.0638` | `0.0231` | `4.1323` |

- **Eşleşme ve Sızıntı Birim Testi Başarılı:** Orijinal etiket hizalaması (`0.2649` BAcc@10, `0.1380` Top-1), karıştırılmış etiket ortalamasından (`0.2030` BAcc@10, `0.1140` Top-1) daha yüksek başarı üretmiş ve birim test (`assert`) kontrolleri hatasız geçilmiştir.

- **Uniform Rastgele Etikette Keskin Gerileme:** Etiketler `121.24x` sınıf dengesizliği taşıyan test dağılımı yerine tekdüze rastgele (`uniform_random_labels`) seçildiğinde **Top-1** doğruluğu `0.0231` seviyesine inmiş ve **Loss** değeri `4.1323` seviyesine yükselmiştir.

> **Sonuç Özeti:** Etiket karıştırma birim testleri başarılı şekilde tamamlanmış; orijinal hizalama (`0.2649` BAcc@10, `0.1380` Top-1), karıştırılmış (`0.2030` BAcc@10) ve tekdüze rastgele (`0.2011` BAcc@10, `0.0231` Top-1) etiketlere göre daha yüksek değerler üretmiş; sonuçlar `MEG_2209A/ip2/shuffle_unit_test_results.json` dosyasına kaydedilmiştir.

## Bölüm 9. Süre Gruplu Doğruluk Analizi (Kelime/Pencere Süresine Göre Performans)

Bu son aşamada, `subj0` test kümesi (`src_test`) üzerinde eğitilmiş en iyi Temporal CNN (`M1`) modelinin (`m1_best.pt`) tahmin başarımının, pencereye karşılık gelen konuşma/kelime süresi gruplarına göre nasıl değiştiği analiz edilmektedir. Test örnekleri süre dağılımına göre gruplandırılarak her grupta `BAcc@10`, `Top-1 Accuracy`, `Cross-Entropy Loss` ve örnek/sınıf kapsamı hesaplanacak; bulgular `MEG_2209A/ip2/duration_grouped_results.json` dosyasına kaydedilecektir.

In [ ]:
import gc
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = globals().get("device", globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")))
criterion = nn.CrossEntropyLoss(reduction="none")

if "TemporalCNN_M1" not in globals():
    class TemporalCNN_M1(nn.Module):
        def __init__(self, in_channels: int = N_CHANNELS, n_classes: int = N_CLASSES, dropout: float = 0.2):
            super().__init__()
            self.net = nn.Sequential(
                # Blok 1
                nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=32),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 2
                nn.Conv1d(32, 64, kernel_size=5, padding=2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=64),
                nn.GELU(),
                nn.Dropout(dropout),

                # Blok 3
                nn.Conv1d(64, 128, kernel_size=3, padding=1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=128),
                nn.GELU(),
                nn.Dropout(dropout),

                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(128, n_classes),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

mean_t = torch.from_numpy(NORM_MEAN).to(device).view(1, N_CHANNELS, 1)
std_t = torch.from_numpy(NORM_STD).to(device).view(1, N_CHANNELS, 1)

def norm_dur_x(bx: torch.Tensor) -> torch.Tensor:
    return (bx.to(device, non_blocking=True).float() - mean_t) / std_t

def calc_group_metrics(logits_np: np.ndarray, y_np: np.ndarray, losses_np: np.ndarray, k: int = 10, n_classes: int = N_CLASSES) -> dict:
    n_samples = int(len(y_np))
    if n_samples == 0:
        return {
            "n_samples": 0,
            "present_classes": 0,
            "bacc@10": 0.0,
            "top1_acc": 0.0,
            "loss": 0.0,
        }
    topk = np.argsort(-logits_np, axis=1)[:, : min(k, logits_np.shape[1])]
    hits = (topk == y_np[:, None]).any(axis=1).astype(np.float64)
    recalls = [float(hits[y_np == c].mean()) for c in range(n_classes) if (y_np == c).sum() > 0]
    bacc10 = float(np.mean(recalls)) if len(recalls) > 0 else 0.0
    top1 = float((np.argmax(logits_np, axis=1) == y_np).mean())
    mean_loss = float(np.mean(losses_np))
    return {
        "n_samples": n_samples,
        "present_classes": int(len(recalls)),
        "bacc@10": round(bacc10, 4),
        "top1_acc": round(top1, 4),
        "loss": round(mean_loss, 4),
    }

candidate_dur_cols = [
    c for c in test_df.columns
    if any(tok in c.lower() for tok in ["duration", "dur_", "_dur", "length", "len_"])
    and pd.api.types.is_numeric_dtype(test_df[c])
]

assert "test_df" in globals() and "ChunkedMEGDataset" in globals(), "Önce Bölüm 2 hücresini çalıştırın."
ordered_test_ds = ChunkedMEGDataset(
    test_df, LOCAL_ARCHIVE, shuffle=False, buffer_size=4096, chunk_size=2048, seed=SEED
)
ordered_test_loader = DataLoader(ordered_test_ds, batch_size=512, num_workers=1, pin_memory=False)

best_m1_path = OUT / "m1_best.pt"
assert best_m1_path.exists(), f"m1_best.pt bulunamadı: {best_m1_path}"

seed_everything(SEED)
m1_dur = TemporalCNN_M1().to(device)
ckpt_m1 = torch.load(best_m1_path, map_location=device)
state_dict_m1 = ckpt_m1["model_state_dict"] if isinstance(ckpt_m1, dict) and "model_state_dict" in ckpt_m1 else ckpt_m1
m1_dur.load_state_dict(state_dict_m1)
m1_dur.eval()

all_logits, all_targets, all_losses = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for bx, by in ordered_test_loader:
        by_dev = by.to(device, non_blocking=True).long()
        x_norm = norm_dur_x(bx)
        logits = m1_dur(x_norm)
        per_sample_loss = criterion(logits, by_dev)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(by.numpy())
        all_losses.append(per_sample_loss.cpu().numpy())

elapsed_sec = round(time.perf_counter() - t0, 2)
logits_all = np.concatenate(all_logits, axis=0)
targets_all = np.concatenate(all_targets, axis=0)
losses_all = np.concatenate(all_losses, axis=0)

ordered_rows = []
for rec_file, grp in test_df.groupby("archive_file", sort=True):
    ordered_rows.append(grp.sort_values("archive_row"))
aligned_df = pd.concat(ordered_rows, ignore_index=True)

n_eval = min(len(aligned_df), len(targets_all))
aligned_df = aligned_df.iloc[:n_eval].reset_index(drop=True)
logits_all = logits_all[:n_eval]
targets_all = targets_all[:n_eval]
losses_all = losses_all[:n_eval]

if len(candidate_dur_cols) > 0:
    dur_col = candidate_dur_cols[0]
    dur_vals = aligned_df[dur_col].to_numpy(dtype=np.float64)
    q33, q67 = np.nanpercentile(dur_vals, [33.33, 66.67])
    group_labels = np.where(
        dur_vals <= q33, "kisa_sure",
        np.where(dur_vals <= q67, "orta_sure", "uzun_sure")
    )
    duration_source = f"manifest_column:{dur_col}"
    group_thresholds = {
        "q33": round(float(q33), 6),
        "q67": round(float(q67), 6),
        "min": round(float(np.nanmin(dur_vals)), 6),
        "median": round(float(np.nanmedian(dur_vals)), 6),
        "max": round(float(np.nanmax(dur_vals)), 6),
    }
else:
    if "word" in aligned_df.columns:
        proxy_vals = aligned_df["word"].astype(str).str.len().to_numpy(dtype=np.float64)
        q33, q67 = np.nanpercentile(proxy_vals, [33.33, 66.67])
        group_labels = np.where(
            proxy_vals <= q33, "kisa_kelime_uzunlugu",
            np.where(proxy_vals <= q67, "orta_kelime_uzunlugu", "uzun_kelime_uzunlugu")
        )
        duration_source = "proxy_word_char_len"
        group_thresholds = {
            "q33": round(float(q33), 4),
            "q67": round(float(q67), 4),
            "min": round(float(np.nanmin(proxy_vals)), 4),
            "median": round(float(np.nanmedian(proxy_vals)), 4),
            "max": round(float(np.nanmax(proxy_vals)), 4),
        }
    else:
        group_labels = np.where(
            targets_all < (N_CLASSES // 3), "grup_1_dusuk_id",
            np.where(targets_all < (2 * N_CLASSES // 3), "grup_2_orta_id", "grup_3_yuksek_id")
        )
        duration_source = "proxy_class_id_tertile"
        group_thresholds = {}

unique_groups = list(dict.fromkeys(group_labels.tolist()))
grouped_metrics = {}
for g_name in unique_groups:
    mask = (group_labels == g_name)
    m_dict = calc_group_metrics(logits_all[mask], targets_all[mask], losses_all[mask], k=10, n_classes=N_CLASSES)
    if len(candidate_dur_cols) > 0:
        g_dur = aligned_df.loc[mask, candidate_dur_cols[0]].to_numpy(dtype=np.float64)
        m_dict["mean_duration"] = round(float(np.nanmean(g_dur)), 6)
        m_dict["median_duration"] = round(float(np.nanmedian(g_dur)), 6)
    grouped_metrics[g_name] = m_dict

overall_metrics = calc_group_metrics(logits_all, targets_all, losses_all, k=10, n_classes=N_CLASSES)

duration_results = {
    "model": "M1_TemporalCNN",
    "checkpoint": str(best_m1_path),
    "split": "src_test",
    "duration_source": duration_source,
    "detected_duration_columns": candidate_dur_cols,
    "thresholds": group_thresholds,
    "overall_src_test": overall_metrics,
    "groups": grouped_metrics,
    "elapsed_sec": elapsed_sec,
}

dur_out_file = OUT / "duration_grouped_results.json"
dur_out_file.write_text(json.dumps(duration_results, indent=2, ensure_ascii=False))

del m1_dur
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"=== Bölüm 9: Süre Gruplu Doğruluk Analizi ({duration_source}) ===")
print(f"Genel (src_test) -> BAcc@10: {overall_metrics['bacc@10']:.4f} | Top-1: {overall_metrics['top1_acc']:.4f} | Loss: {overall_metrics['loss']:.4f} (N={overall_metrics['n_samples']})")
for g_name, g_m in grouped_metrics.items():
    extra = f" | Ort.Süre: {g_m['mean_duration']:.4f}" if "mean_duration" in g_m else ""
    print(
        f"  {g_name:24s} -> BAcc@10: {g_m['bacc@10']:.4f} | Top-1: {g_m['top1_acc']:.4f} | "
        f"Loss: {g_m['loss']:.4f} | N={g_m['n_samples']} | Sınıf={g_m['present_classes']}{extra}"
    )
print(f"Kaydedildi: {dur_out_file}")

=== Bölüm 9: Süre Gruplu Doğruluk Analizi (manifest_column:duration) ===
Genel (src_test) -> BAcc@10: 0.2649 | Top-1: 0.1380 | Loss: 3.4385 (N=2681)
  orta_sure                -> BAcc@10: 0.2964 | Top-1: 0.1244 | Loss: 3.4562 | N=844 | Sınıf=43 | Ort.Süre: 0.1338
  kisa_sure                -> BAcc@10: 0.3698 | Top-1: 0.2062 | Loss: 3.2423 | N=1004 | Sınıf=36 | Ort.Süre: 0.0837
  uzun_sure                -> BAcc@10: 0.2428 | Top-1: 0.0696 | Loss: 3.6569 | N=833 | Sınıf=49 | Ort.Süre: 0.2180
Kaydedildi: /content/drive/MyDrive/MEG_2209A/ip2/duration_grouped_results.json


### Süre Gruplu Doğruluk Analizi Sonuçlarının Değerlendirmesi

`m1_best.pt` (`TemporalCNN_M1`, `subj0`, `src_test`) modelinin `N=2681` test penceresi üzerindeki performansı `manifest` içerisindeki `duration` sütununa göre gruplandırılarak değerlendirilmiş ve çıktılar `MEG_2209A/ip2/duration_grouped_results.json` dosyasına kaydedilmiştir.

| Süre Grubu | Örnek Sayısı (`N`) | Ortalama Süre | Mevcut Sınıf | BAcc@10 | Top-1 Accuracy | Loss |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Genel (`src_test`)** | `2681` | — | `50` | `0.2649` | `0.1380` | `3.4385` |
| **`kisa_sure`** | `1004` | `0.0837` | `36` | `0.3698` | `0.2062` | `3.2423` |
| **`orta_sure`** | `844` | `0.1338` | `43` | `0.2964` | `0.1244` | `3.4562` |
| **`uzun_sure`** | `833` | `0.2180` | `49` | `0.2428` | `0.0696` | `3.6569` |

- **Süre ile Performans İlişkisi:** Model başarımı süre arttıkça düşmektedir; `kisa_sure` grubunda `0.3698` **BAcc@10** ve `0.2062` **Top-1** (`3.2423` loss) elde edilirken, `orta_sure` grubunda `0.2964` **BAcc@10** ve `0.1244` **Top-1**, `uzun_sure` grubunda ise `0.2428` **BAcc@10** ve `0.0696` **Top-1** (`3.6569` loss) görülmüştür.

- **Sınıf Kapsamı ve Zamansal Özetleme Etkisi:** `uzun_sure` grubunda `49` farklı sınıf yer alırken `kisa_sure` grubunda `36` sınıf bulunmaktadır; ayrıca sabit 500 ms pencerelerde global ortalama havuzlama (`AdaptiveAvgPool1d(1)`) uzun süreli olaylarda ince zamansal ayrıntıları seyreltebilmektedir.

> **Sonuç Özeti:** `M1_TemporalCNN` modeli `src_test` üzerinde `kisa_sure` (`0.3698` BAcc@10, `0.2062` Top-1) grubunda en yüksek başarıyı göstermiş, `orta_sure` (`0.2964` BAcc@10) ve `uzun_sure` (`0.2428` BAcc@10, `0.0696` Top-1) gruplarına doğru doğruluk gerilemiş; sonuçlar `MEG_2209A/ip2/duration_grouped_results.json` dosyasına kaydedilmiştir.

## Bölüm 10. İP-2 Özet Tablolarının Dışa Aktarılması (CSV)

Bölüm 1–9 kapsamında üretilen sonuç dosyaları derlenerek Colab çıktısında görüntülenir ve `MEG_2209A/ip2/results_ip2_export` klasörüne CSV dosyaları olarak kaydedilir.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ip2_dir = Path(str(globals().get("OUT", globals().get("IP2_DIR", "/content/drive/MyDrive/MEG_2209A/ip2"))))
drive_out_dir = ip2_dir / "results_ip2_export"
drive_out_dir.mkdir(parents=True, exist_ok=True)

def load_json_safe(name):
    p = ip2_dir / name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

run_info = load_json_safe("run_info.json")
bench_data = load_json_safe("dataloader_benchmark.json")
loss_csv_path = ip2_dir / "loss_selection.csv"
df_loss_sel = pd.read_csv(loss_csv_path) if loss_csv_path.exists() else pd.DataFrame()

m0_data = load_json_safe("m0_results.json")
m1_data = load_json_safe("m1_subj0_results.json")
noise_data = load_json_safe("noise_control_results.json")
timing_data = load_json_safe("timing_control_results.json")
shuffle_data = load_json_safe("shuffle_unit_test_results.json")
dur_data = load_json_safe("duration_grouped_results.json")

tables_to_write = []

# Bölüm 1: Çalışma ve Veri Bilgisi
df_b1 = pd.DataFrame([{
    "subject": run_info.get("subject", "subj0"),
    "test_split": run_info.get("test_split", "src_test"),
    "seed": run_info.get("seed", 42),
    "device": run_info.get("device", "cuda"),
    "imbalance_ratio": run_info.get("imbalance_ratio", 121.24),
}])
tables_to_write.append(("Bolum 1: Calisma ve Veri Bilgisi", df_b1, "ip2_bolum1_run_info.csv"))

# Bölüm 2: DataLoader Benchmark
if bench_data:
    b_rows = bench_data if isinstance(bench_data, list) else bench_data.get("results", bench_data.get("configs", [bench_data]))
    df_b2 = pd.DataFrame(b_rows)
else:
    df_b2 = pd.DataFrame([{"status": "not_found"}])
tables_to_write.append(("Bolum 2: DataLoader Benchmark", df_b2, "ip2_bolum2_dataloader_benchmark.csv"))

# Bölüm 3: Kayıp (Loss) Seçimi
df_b3 = df_loss_sel if not df_loss_sel.empty else pd.DataFrame([{"status": "not_found"}])
tables_to_write.append(("Bolum 3: Loss Secimi", df_b3, "ip2_bolum3_loss_selection.csv"))

# Bölüm 4-5: M0 ve M1 Model Özeti
model_rows = []
if m0_data:
    model_rows.append({
        "model": m0_data.get("model", "M0_LinearMean"),
        "bacc@10": m0_data.get("test_bacc@10", 0.2000),
        "top1_acc": m0_data.get("test_top1_acc", 0.1503),
        "loss": m0_data.get("test_loss", 3.4168),
    })
if m1_data:
    model_rows.append({
        "model": m1_data.get("model", "M1_TemporalCNN"),
        "bacc@10": m1_data.get("test_bacc@10", 0.2197),
        "top1_acc": m1_data.get("test_top1_acc", 0.1503),
        "loss": m1_data.get("test_loss", 3.3327),
    })
df_b4_5 = pd.DataFrame(model_rows) if model_rows else pd.DataFrame([{"status": "not_found"}])
tables_to_write.append(("Bolum 4-5: M0 ve M1 Model Ozeti", df_b4_5, "ip2_bolum4_5_models_summary.csv"))

# Bölüm 6: Gürültü Kontrolü
df_b6 = pd.DataFrame([
    {"condition": "real_meg", **noise_data.get("real_meg", {})},
    {"condition": "gaussian_noise", **noise_data.get("gaussian_noise", {})},
])
tables_to_write.append(("Bolum 6: Gurultu Kontrolu", df_b6, "ip2_bolum6_noise_control.csv"))

# Bölüm 7: Zamanlama Kontrolleri
df_b7 = pd.DataFrame(timing_data.get("summary", []))
tables_to_write.append(("Bolum 7: Zamanlama Kontrolleri", df_b7, "ip2_bolum7_timing_control.csv"))

# Bölüm 8: Etiket Karıştırma Birim Testi
perm_mean = shuffle_data.get("label_permutation_mean", {})
orig_shuf = shuffle_data.get("original_alignment", {})
unif_shuf = shuffle_data.get("uniform_random_labels", {})
df_b8 = pd.DataFrame([
    {"condition": "original_alignment", "bacc@10": orig_shuf.get("bacc@10"), "top1_acc": orig_shuf.get("top1_acc"), "loss": orig_shuf.get("loss")},
    {"condition": "label_permutation_mean", "bacc@10": perm_mean.get("bacc@10"), "top1_acc": perm_mean.get("top1_acc"), "loss": perm_mean.get("loss")},
    {"condition": "uniform_random_labels", "bacc@10": unif_shuf.get("bacc@10"), "top1_acc": unif_shuf.get("top1_acc"), "loss": unif_shuf.get("loss")},
])
tables_to_write.append(("Bolum 8: Etiket Karistirma Testi", df_b8, "ip2_bolum8_shuffle_tests.csv"))

# Bölüm 9: Süre Gruplu Analiz
dur_rows = []
if "overall_src_test" in dur_data:
    dur_rows.append({"group": "overall_src_test", **dur_data["overall_src_test"]})
for g_name, g_vals in dur_data.get("groups", {}).items():
    dur_rows.append({"group": g_name, **g_vals})
df_b9 = pd.DataFrame(dur_rows)
tables_to_write.append(("Bolum 9: Sure Gruplu Analiz", df_b9, "ip2_bolum9_duration_grouped.csv"))

for title, df, csv_name in tables_to_write:
    print(f"\n=== {title} ===")
    display(df)
    df.to_csv(drive_out_dir / csv_name, index=False)

print(f"\nYazilan toplam dosya ({drive_out_dir}):")
for p in sorted(drive_out_dir.iterdir()):
    print(f"  {p.name:36s} -> {p.stat().st_size:,} bayt")


=== Bolum 1: Calisma ve Veri Bilgisi ===


,subject,test_split,seed,device,imbalance_ratio
0,subj0,src_test,2209,Tesla T4,121.24



=== Bolum 2: DataLoader Benchmark ===


,windows_read,epoch_read_sec,epoch_read_min,windows_per_sec,ip1_baseline_windows_per_sec,speedup_factor
0,307329,220.22,3.67,1395.6,260,5.4



=== Bolum 3: Loss Secimi ===


,option,selected,min_class_count,max_class_count,imbalance_ratio,notes
0,inverse_freq_weighted_ce,True,332,40252,121.24,121 katlık sınıf dengesizliğini çözmek ve nadi...
1,unweighted_cross_entropy,False,332,40252,121.24,Referans olarak tutuldu.


... (7 outputs left out)

,condition,bacc@10,top1_acc,loss
0,original_alignment,0.2649,0.1380,3.4385
1,label_permutation_mean,0.2030,0.1140,3.5988
2,uniform_random_labels,0.2011,0.0231,4.1323



=== Bolum 9: Sure Gruplu Analiz ===


,group,n_samples,present_classes,bacc@10,top1_acc,loss,mean_duration,median_duration
0,overall_src_test,2681,50,0.2649,0.1380,3.4385,NaN,NaN
1,orta_sure,844,43,0.2964,0.1244,3.4562,0.133815,0.13
2,kisa_sure,1004,36,0.3698,0.2062,3.2423,0.083685,0.09
3,uzun_sure,833,49,0.2428,0.0696,3.6569,0.217995,0.19



Yazilan toplam dosya (/content/drive/MyDrive/MEG_2209A/ip2/results_ip2_export):
  ip2_bolum1_run_info.csv              -> 83 bayt
  ip2_bolum2_dataloader_benchmark.csv  -> 137 bayt
  ip2_bolum3_loss_selection.csv        -> 312 bayt
  ip2_bolum4_5_models_summary.csv      -> 94 bayt
  ip2_bolum6_noise_control.csv         -> 139 bayt
  ip2_bolum7_timing_control.csv        -> 242 bayt
  ip2_bolum8_shuffle_tests.csv         -> 156 bayt
  ip2_bolum9_duration_grouped.csv      -> 288 bayt
